# Unemployment Analysis in India — CodeAlpha Task 2

Professional exploratory analysis of monthly unemployment, labour participation, regional variation, and the March–June 2020 COVID shock.

**Important:** the uploaded file is CSV data with an `.xls` filename. The notebook reads it as UTF-8 CSV.

In [ ]:
from pathlib import Path
import glob
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 14, 'axes.titleweight': 'bold'})

# Works locally and on Kaggle after attaching the dataset.
candidates = [
    Path('/kaggle/input/UnemploymentinIndia.xls'),
    Path('/kaggle/input/unemployment-in-india/Unemployment in India.csv'),
    Path('../data_raw/UnemploymentinIndia.csv'),
    Path('data_raw/UnemploymentinIndia.csv'),
]
files = [p for p in candidates if p.exists()]
if not files:
    files = [Path(p) for p in glob.glob('/kaggle/input/**/*', recursive=True) if Path(p).is_file() and Path(p).suffix.lower() in {'.csv', '.xls'}]
if not files:
    raise FileNotFoundError('Attach the unemployment dataset or place it in data_raw/.')
DATA_PATH = files[0]
print('Using:', DATA_PATH)
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df.columns = df.columns.str.strip()
df = df.dropna(how='all').copy()
print('Raw rows after removing blank rows:', len(df))


In [ ]:
RATE = 'Estimated Unemployment Rate (%)'
EMPLOYED = 'Estimated Employed'
LPR = 'Estimated Labour Participation Rate (%)'

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
for col in [RATE, EMPLOYED, LPR]:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['Region'] = df['Region'].astype('string').str.strip()
df['Area'] = df['Area'].astype('string').str.strip()
df = df.dropna(subset=['Date', RATE, 'Region', 'Area']).sort_values('Date')
df['Period'] = df['Date'].map(lambda x: 'Pre-COVID' if x < pd.Timestamp('2020-03-01') else 'COVID shock')
df['Month'] = df['Date'].dt.to_period('M').dt.to_timestamp()

period_summary = df.groupby('Period', observed=True)[RATE].agg(['count','mean','median','min','max']).round(2)
area_summary = df.groupby(['Period','Area'], observed=True)[RATE].mean().round(2).unstack()
region_summary = df.groupby('Region')[RATE].mean().sort_values(ascending=False).round(2)
print('Clean shape:', df.shape)
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Regions:', df['Region'].nunique())
display(period_summary)
print('Top five regions by average unemployment rate:')
display(region_summary.head())


## COVID-19 impact

The dataset covers May 2019–June 2020. I use May 2019–February 2020 as the pre-COVID baseline and March–June 2020 as the COVID shock window. This is a descriptive before/after comparison, not a causal estimate.


In [ ]:
monthly = df.groupby('Month', as_index=False)[RATE].mean()
monthly_area = df.groupby(['Month','Area'], as_index=False)[RATE].mean()
pre = df.loc[df['Period'].eq('Pre-COVID'), RATE].mean()
shock = df.loc[df['Period'].eq('COVID shock'), RATE].mean()
print(f'Pre-COVID mean: {pre:.2f}%')
print(f'COVID shock mean: {shock:.2f}%')
print(f'Increase: {(shock/pre-1)*100:.1f}%')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
ax = axes[0,0]
ax.plot(monthly['Month'], monthly[RATE], color='#0F766E', marker='o', linewidth=3, label='All areas')
for area, color in [('Urban','#F97316'),('Rural','#2563EB')]:
    temp = monthly_area[monthly_area['Area'].eq(area)]
    ax.plot(temp['Month'], temp[RATE], color=color, marker='o', label=area)
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-30'), color='#FDE68A', alpha=.45, label='COVID shock')
ax.set_title('Monthly unemployment trend')
ax.set_ylabel('Rate (%)'); ax.legend(frameon=False)

ax = axes[0,1]
values = [pre, shock]
bars = ax.bar(['Pre-COVID','COVID shock'], values, color=['#94A3B8','#DC2626'])
ax.bar_label(bars, labels=[f'{v:.1f}%' for v in values], padding=3, fontweight='bold')
ax.set_title('Average rate comparison'); ax.set_ylabel('Rate (%)')

ax = axes[1,0]
rank = region_summary.sort_values()
ax.barh(rank.index, rank.values, color='#0F766E')
ax.set_title('Regional average unemployment'); ax.set_xlabel('Rate (%)')

ax = axes[1,1]
for period, color in [('Pre-COVID','#64748B'),('COVID shock','#DC2626')]:
    temp = df[df['Period'].eq(period)]
    ax.scatter(temp[LPR], temp[RATE], s=20, alpha=.4, color=color, label=period)
ax.set_title('Participation vs unemployment')
ax.set_xlabel('Labour participation rate (%)'); ax.set_ylabel('Unemployment rate (%)'); ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Key findings and policy implications

- The national average unemployment rate rises sharply during March–June 2020, with the monthly peak in May 2020.
- Regional averages differ materially, so national targeting alone would miss local labour-market stress.
- Urban and rural unemployment both rise; the two segments should be monitored separately.
- Labour participation should be interpreted alongside unemployment to distinguish job loss from labour-force exit.
- The evidence supports geographically targeted employment support, high-frequency monitoring, and post-shock evaluation using longer time series.

**Limitation:** the supplied data ends in June 2020, so this notebook identifies the initial COVID shock but cannot evaluate the full recovery.
